# Microsoft Agent Framework — Azure OpenAI (Responses API)

In diesem Codebeispiel verwenden Sie das **Microsoft Agent Framework (MAF)**, um einen einfachen Agenten zu erstellen, der von **Azure OpenAI** unterstützt wird, unter Verwendung der **Responses API**.

> **Migrationshinweis:** Dieses Beispiel verwendete zuvor Semantic Kernel mit GitHub Models. Es wurde auf das Microsoft Agent Framework migriert, und GitHub Models (veraltet, wird im Juli 2026 eingestellt) wurde durch Azure OpenAI ersetzt, das die Responses API unterstützt. Der `OpenAIChatClient` in MAF zielt standardmäßig auf den stabilen `/openai/v1/` Endpunkt von Azure OpenAI ab und verwendet die Responses API.

Ziel dieses Beispiels ist es, die Schritte zu demonstrieren, die später in weiteren Codebeispielen zur Implementierung verschiedener agentischer Muster angewendet werden.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importieren der benötigten Python-Pakete


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definition eines Werkzeugs

Im Microsoft Agent Framework ist ein **Werkzeug** eine einfache Python-Funktion, die mit `@tool` dekoriert ist und vom Agenten aufgerufen werden kann. Unten definieren wir ein Werkzeug, das ein zufälliges Urlaubsziel zurückgibt und vermeidet, das vorherige zu wiederholen.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Erstellung des Agenten

Hier erstellen wir den Agenten namens `TravelAgent`.

In diesem Beispiel verwenden wir sehr grundlegende Anweisungen. Sie können diese Anweisungen gerne anpassen, um zu beobachten, wie sich das Verhalten des Agenten verändert.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Den Agenten ausführen

Nun können wir den Agenten ausführen. Wir erstellen eine `AgentSession`, damit sich der Agent die Unterhaltung über mehrere Züge hinweg merkt, und senden dann zwei `user_inputs`. Das erste fragt nach einer Reise; das zweite sagt, dass dem Nutzer der Vorschlag nicht gefallen hat und bittet um einen weiteren — der Agent nutzt den Sitzungsverlauf plus das Tool `get_random_destination`, um zu antworten.

Du kannst diese Nachrichten anpassen, um zu beobachten, wie der Agent unterschiedlich reagiert. Antworten werden **Token für Token** gestreamt.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
